In [6]:
import math
import random
from typing import Callable

import scipy.stats as st
import numpy as np

def est_D(probabilidades: list[float]):
    n = len(probabilidades)
    prob = sorted(probabilidades)

    D_p = [((i + 1)/n - prob[i]) for i in range(n)]  # Fe(Y) - F(Y)
    D_m = [(prob[i] - i/n) for i in range(n)]  # F(Y) - Fe(Y)

    d = max(max(D_p), max(D_m))

    return d


def estimar_p_valor_simulaciones_continua(
    d_obs: float,
    gen_prob: Callable[[], list[float]],
    N_SIM: int = 10_000
):
    # Se utiliza el estadístico de Kolmogorov-Smirnov

    p_valor = 0.0
    for _ in range(N_SIM):
        probabilidades = gen_prob()
        d_j = est_D(probabilidades)

        if d_j >= d_obs:
            p_valor += 1

    p_valor = p_valor / N_SIM
    return p_valor

### Ejercicio 8.
Se sortean elementos de un conjunto de datos que tiene una distribución t-student de $11$ grados de libertad. El investigador, que no conoce la forma verdadera de la distribución, asume que la misma es normal. <br>
Analice la validez de esta suposición para muestras de tamaños $10$, $20$, $100$ y $1000$ elementos Para ello realice simulaciones numéricas e implemente el test de Kolmogorov-Smirnov a los datos simulados, asumiendo una distribución $N(0,1)$. Presente los resultados en una tabla que contenga el número de elementos de la simulación, el valor del estadístico $D$ y el *p−valor*

**Ayuda**: Función de probabilidad normal: Para obtener la función de probabilidad normal, se puede usar la función $math.erf$. Por ejemplo, la cantidad $math.erf(x/math.sqrt(2.))/2.+0.5$ equivale a

$$ \int_{- \infty}^{x} N(0, 1)(t) dt $$

**Ayuda**: Generación de números aleatorios con una distribución t-student: Utilice el siguiente código para generar números aleatorios que siguen una distribución T-student:

```python
import math
import random

def rt(df): # df grados de libertad
    x = random.gauss(0.0, 1.0)
    y = 2.0*random.gammavariate(0.5*df, 2.0)
    return x / (math.sqrt(y/df))
```


---

### Análisis
$$
\begin{array}{cl}
    H_0 & = \text{"Los datos provienen de una distribución Normal estándar"} \\
    H_1 & = \text{"Los datos no provienen de una distribución Normal estándar"}
\end{array}
$$

Asumo $\alpha = 0.05$

In [7]:
def gen_dist_t_student(df):
    # df grados de libertad

    x = random.gauss(0.0, 1.0)
    y = 2.0 * random.gammavariate(0.5 * df, 2.0)

    return x / (math.sqrt(y/df))

def simulaciones(n: int, N_SIM: int = 10_000):
    muestra = [gen_dist_t_student(11) for _ in range(n)]

    prob = [st.norm.cdf(muestra[i], loc=0, scale=1) for i in range(n)]
    d_obs = est_D(prob)

    p_valor = estimar_p_valor_simulaciones_continua(
        d_obs,
        lambda: np.random.uniform(0, 1, n)
    )

    return d_obs, p_valor

n = 10
d_obs, p_valor = simulaciones(n)
print(f"Con una muestra de tamaño {n} se obtuvieron:")
print(f" - d_obs: ", d_obs)
print(f" - p_valor: ", p_valor)
print()

n = 20
d_obs, p_valor = simulaciones(n)
print(f"Con una muestra de tamaño {n} se obtuvieron:")
print(f" - d_obs: ", d_obs)
print(f" - p_valor: ", p_valor)
print()

n = 100
d_obs, p_valor = simulaciones(n)
print(f"Con una muestra de tamaño {n} se obtuvieron:")
print(f" - d_obs: ", d_obs)
print(f" - p_valor: ", p_valor)
print()

n = 1_000
d_obs, p_valor = simulaciones(n)
print(f"Con una muestra de tamaño {n} se obtuvieron:")
print(f" - d_obs: ", d_obs)
print(f" - p_valor: ", p_valor)

Con una muestra de tamaño 10 se obtuvieron:
 - d_obs:  0.23733770141750515
 - p_valor:  0.554

Con una muestra de tamaño 20 se obtuvieron:
 - d_obs:  0.2639862101075877
 - p_valor:  0.1043

Con una muestra de tamaño 100 se obtuvieron:
 - d_obs:  0.13475217257594788
 - p_valor:  0.0505

Con una muestra de tamaño 1000 se obtuvieron:
 - d_obs:  0.07280067346117466
 - p_valor:  0.0


A medida de que la muestra aumenta, el *p-valor* va tendiendo a ser mas pequeño que $\alpha$, por lo que para una pequeña cantidad de muestras la $H_0$ no se rechazaría (no hay suficiente evidencia para rechazarla), pero a medida de que aumente las muestras si.